In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [3]:
# =========================================
# 🚀 FINAL CLEAN PIPELINE (ALL FIXED)
# =========================================

import numpy as np
import pandas as pd
import optuna

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

print("🚀 Start")

# =========================================
# LOAD
# =========================================
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

target = 'Irrigation_Need'

X = train.drop(columns=[target, 'id'])
y = train[target]

X_test = test.drop(columns=['id'])
test_ids = test['id']

# =========================================
# TARGET ENCODE
# =========================================
le = LabelEncoder()
y = le.fit_transform(y)

# =========================================
# FEATURE ENGINEERING
# =========================================
def fe(df):
    df['ET0'] = df['Temperature_C'] * df['Wind_Speed_kmh'] * df['Sunlight_Hours'] / (df['Humidity'] + 1)
    df['water_balance'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm'] - df['ET0']
    df['stress'] = df['Temperature_C'] * (100 - df['Humidity']) / (df['Soil_Moisture'] + 1)
    df['retention'] = df['Soil_Moisture'] * df['Organic_Carbon']
    df['crop_stage'] = df['Crop_Type'] + "_" + df['Crop_Growth_Stage']
    df['region_season'] = df['Region'] + "_" + df['Season']
    return df

X = fe(X)
X_test = fe(X_test)

# =========================================
# 🔥 FIX CATEGORICALS (XGB SAFE)
# =========================================
cat_cols = X.select_dtypes(include=['object', 'category']).columns

for col in cat_cols:
    combined = pd.concat([X[col], X_test[col]])
    codes = combined.astype('category').cat.codes.astype('int32')
    X[col] = codes[:len(X)].values
    X_test[col] = codes[len(X):].values

# FINAL SAFETY CHECK
assert not any(X.dtypes == 'category'), "❌ Category dtype still present!"

print("✅ Encoding fixed")

# =========================================
# CV
# =========================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# =========================================
# CV HELPERS
# =========================================
def cv_lgb(params):
    oof = np.zeros((len(X), 3))
    for tr, val in skf.split(X, y):
        m = lgb.LGBMClassifier(**params, n_estimators=300)
        m.fit(X.iloc[tr], y[tr])
        oof[val] = m.predict_proba(X.iloc[val])
    return accuracy_score(y, oof.argmax(1))

def cv_xgb(params):
    oof = np.zeros((len(X), 3))
    for tr, val in skf.split(X, y):
        m = xgb.XGBClassifier(
            **params,
            n_estimators=300,
            eval_metric='mlogloss',
            use_label_encoder=False
        )
        m.fit(X.iloc[tr], y[tr])
        oof[val] = m.predict_proba(X.iloc[val])
    return accuracy_score(y, oof.argmax(1))

def cv_cat(params):
    oof = np.zeros((len(X), 3))
    for tr, val in skf.split(X, y):
        m = CatBoostClassifier(**params, iterations=300, verbose=0)
        m.fit(X.iloc[tr], y[tr])
        oof[val] = m.predict_proba(X.iloc[val])
    return accuracy_score(y, oof.argmax(1))

# =========================================
# 🔥 LGBM (3 trials)
# =========================================
print("🔍 LGB")

def obj_lgb(t):
    p = {
        "learning_rate": t.suggest_float("learning_rate", 0.03, 0.1),
        "num_leaves": t.suggest_int("num_leaves", 31, 80),
        "max_depth": t.suggest_int("max_depth", 4, 8)
    }
    return cv_lgb(p)

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(obj_lgb, n_trials=3)
lgb_p = study_lgb.best_params

# =========================================
# 🔥 XGB (FIXED PARAM NAMES)
# =========================================
print("🔍 XGB")

def obj_xgb(t):
    p = {
        "learning_rate": t.suggest_float("learning_rate", 0.03, 0.1),
        "max_depth": t.suggest_int("max_depth", 4, 8),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0)
    }
    return cv_xgb(p)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(obj_xgb, n_trials=3)
xgb_p = study_xgb.best_params

# =========================================
# 🔥 CAT (3 trials)
# =========================================
print("🔍 CAT")

def obj_cat(t):
    p = {
        "depth": t.suggest_int("depth", 4, 8),
        "learning_rate": t.suggest_float("learning_rate", 0.03, 0.1)
    }
    return cv_cat(p)

study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(obj_cat, n_trials=3)
cat_p = study_cat.best_params

# =========================================
# 🔥 OOF STACKING
# =========================================
print("🚀 Stacking")

oof_lgb = np.zeros((len(X), 3))
oof_xgb = np.zeros((len(X), 3))
oof_cat = np.zeros((len(X), 3))

test_lgb = np.zeros((len(X_test), 3))
test_xgb = np.zeros((len(X_test), 3))
test_cat = np.zeros((len(X_test), 3))

for i, (tr, val) in enumerate(skf.split(X, y)):
    print(f"Fold {i+1}")

    # LGB
    m = lgb.LGBMClassifier(**lgb_p, n_estimators=600)
    m.fit(X.iloc[tr], y[tr])
    oof_lgb[val] = m.predict_proba(X.iloc[val])
    test_lgb += m.predict_proba(X_test) / 3

    # XGB
    m = xgb.XGBClassifier(**xgb_p, n_estimators=600, eval_metric='mlogloss')
    m.fit(X.iloc[tr], y[tr])
    oof_xgb[val] = m.predict_proba(X.iloc[val])
    test_xgb += m.predict_proba(X_test) / 3

    # CAT
    m = CatBoostClassifier(**cat_p, iterations=600, verbose=0)
    m.fit(X.iloc[tr], y[tr])
    oof_cat[val] = m.predict_proba(X.iloc[val])
    test_cat += m.predict_proba(X_test) / 3

# =========================================
# META MODEL
# =========================================
X_meta = np.hstack([oof_lgb, oof_xgb, oof_cat])
X_test_meta = np.hstack([test_lgb, test_xgb, test_cat])

meta = LogisticRegression(max_iter=1000)
meta.fit(X_meta, y)

print("📊 CV:", accuracy_score(y, meta.predict(X_meta)))

# =========================================
# SUBMISSION
# =========================================
preds = meta.predict(X_test_meta)
preds = le.inverse_transform(preds)

sub = pd.DataFrame({
    "id": test_ids,)
    "Irrigation_Need": preds
})

sub.to_csv("submission.csv", index=False)

print("🎉 DONE")

🚀 Start


[I 2026-04-21 12:18:48,705] A new study created in memory with name: no-name-a4e3d73c-2203-47bb-b943-c8aa85a0d8f5


✅ Encoding fixed
🔍 LGB
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018876 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3757
[LightGBM] [Info] Number of data points in the train set: 420000, number of used features: 25
[LightGBM] [Info] Start training from score -3.400769
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968945
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] 

[I 2026-04-21 12:21:05,694] Trial 0 finished with value: 0.984315873015873 and parameters: {'learning_rate': 0.09672982695226638, 'num_leaves': 63, 'max_depth': 6}. Best is trial 0 with value: 0.984315873015873.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019081 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3757
[LightGBM] [Info] Number of data points in the train set: 420000, number of used features: 25
[LightGBM] [Info] Start training from score -3.400769
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968945
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018968 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3757
[LightGBM] [Info] Number of data points in the train set: 420000, number of used features: 25
[LightGBM] [Info] Start training from score -3.400769
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Star

...

🔍 XGB


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:24:48] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:25:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:26:19] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
[I 2026-04-21 12:27:04,617] Trial 0 finished with value: 0.9846111111111111 and parameters: {'learning_rate': 0.09043399130037058, 'max_depth': 8, 'subsample': 0.7573921576254364, 'colsample_bytree': 0.8783590428330385}. Best is trial 0 with value: 0.9846111111111111.
/usr/local/lib/python3.12/

🔍 CAT


[I 2026-04-21 12:32:32,513] Trial 0 finished with value: 0.9842603174603175 and parameters: {'depth': 5, 'learning_rate': 0.06376409896176417}. Best is trial 0 with value: 0.9842603174603175.
[I 2026-04-21 12:35:05,414] Trial 1 finished with value: 0.984231746031746 and parameters: {'depth': 6, 'learning_rate': 0.0389866695400016}. Best is trial 0 with value: 0.9842603174603175.
[I 2026-04-21 12:37:10,685] Trial 2 finished with value: 0.984215873015873 and parameters: {'depth': 4, 'learning_rate': 0.07748805542256909}. Best is trial 0 with value: 0.9842603174603175.


🚀 Stacking
Fold 1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018973 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3757
[LightGBM] [Info] Number of data points in the train set: 420000, number of used features: 25
[LightGBM] [Info] Start training from score -3.400769
[LightGBM] [Info] Start training from score -0.532443
[LightGBM] [Info] Start training from score -0.968945
Fold 2
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.020020 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3757
[LightGBM] [Info] Number of data points in the train set: 420000, number of used features: 25
[LightGBM] [Info] Start training from score -3.400769
[LightGBM] [Info] Start training from score -0.5324